In [27]:
import pandas as pd

df = pd.read_csv('../data/sales_history.csv')

df.isna().sum()

df_test = df.copy()

In [28]:
df_test.fillna(0, inplace=True)

df_ordenat = df_test.sort_values(by=['channel','product_line', 'region', 'month'], ascending=True)
conteo = df_ordenat.groupby(['channel','product_line', 'region'])['month'].size().reset_index(name='counts')
df_ordenat = df_ordenat.sort_values(["channel", "product_line", "region", "month"]).copy()

grup = df_ordenat.groupby(["channel", "product_line", "region"])["sales_units"]

train = df_ordenat[df_ordenat["month"].between("2022-01", "2024-12")].copy()
val = df_ordenat[df_ordenat["month"].between("2025-01", "2025-12")].copy()
test = df_ordenat[df_ordenat["month"].between("2026-01", "2026-06")].copy()

for nombre, datos in [("Entrenamiento", train), ("Validación", val), ("Prueba", test)]:
    print(nombre, len(datos), datos["month"].min(), datos["month"].max())

grupo_media = (train.groupby(["channel", "product_line", "region"])["sales_units"].mean().rename("pred_media").reset_index())

claves = ["channel", "product_line", "region"]

# Añadimos a cada mes las ventas de esa serie 12 meses antes.
df_con_lag = df_ordenat.copy()
df_con_lag["pred_año_anterior"] = (df_con_lag.groupby(claves)["sales_units"].shift(12))

# Nos quedamos con los meses de validación: 2025.
val_comparar = df_con_lag[df_con_lag["month"].between("2025-01", "2025-12")].copy()

# A cada fila le añadimos la media de entrenamiento de su grupo.
comparacion = val_comparar.merge(grupo_media,on=claves,how="left")

comparacion[["month", *claves, "sales_units", "pred_media", "pred_año_anterior"]].head(12)

Entrenamiento 648 2022-01 2024-12
Validación 216 2025-01 2025-12
Prueba 108 2026-01 2026-06


,month,channel,product_line,region,sales_units,pred_media,pred_año_anterior
0,2025-01,ecommerce,electric,coast,60,70.222222,55.0
1,2025-02,ecommerce,electric,coast,76,70.222222,75.0
2,2025-03,ecommerce,electric,coast,57,70.222222,82.0
3,2025-04,ecommerce,electric,coast,121,70.222222,118.0
4,2025-05,ecommerce,electric,coast,83,70.222222,90.0
5,2025-06,ecommerce,electric,coast,109,70.222222,68.0
6,2025-07,ecommerce,electric,coast,102,70.222222,92.0
7,2025-08,ecommerce,electric,coast,104,70.222222,72.0
8,2025-09,ecommerce,electric,coast,81,70.222222,79.0
9,2025-10,ecommerce,electric,coast,76,70.222222,59.0


In [29]:
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split

train["month"] = pd.to_datetime(train["month"], format="%Y-%m")
train["year"] = train["month"].dt.year
train["month_num"] = train["month"].dt.month
train = train.drop(columns=["month"])

val["month"] = pd.to_datetime(val["month"], format="%Y-%m")
val["year"] = val["month"].dt.year
val["month_num"] = val["month"].dt.month
val = val.drop(columns=["month"])

columnas_a_quitar = [
    "sales_units",
    "closing_stock_units",
    "revenue",
    "stockout",
    "website_sessions",
    "marketing_spend",
    "avg_discount",
]

X = train.drop(columns=columnas_a_quitar)
y = train["sales_units"]

X_2025 = val.drop(columns=columnas_a_quitar)
y_2025 = val["sales_units"]

X = pd.get_dummies(X, drop_first=False, dtype=int)
X_2025 = pd.get_dummies(X_2025, drop_first=False, dtype=int)
X_2025 = X_2025.reindex(columns=X.columns, fill_value=0)

model = LinearRegression()
model.fit(X, y)

prediccion = model.predict(X_2025)


# 1. Regresio Lineal


In [30]:
import numpy as np
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

model_lineal = LinearRegression()
model_lineal.fit(X, y)

prediccion = model_lineal.predict(X_2025)

print("Ventas reales:", y_2025.to_numpy()[:12])
print("Predicciones: ", prediccion[:12].round(2))

print("MAE:", mean_absolute_error(y_2025, prediccion))
print("RMSE:", np.sqrt(mean_squared_error(y_2025, prediccion)))
print("R²:", r2_score(y_2025, prediccion))

Ventas reales: [ 60  76  57 121  83 109 102 104  81  76  68  68]
Predicciones:  [ 70.95  78.81  57.32 102.45  87.49  85.82  98.73  86.22  82.92  64.23
  77.01  75.59]
MAE: 13.619895808136313
RMSE: 18.308653410662686
R²: 0.8471975837702336


# 2. Decision Tree Regressor

In [31]:
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import mean_absolute_error

model_Decision_Tree = DecisionTreeRegressor(random_state=1)
model_Decision_Tree.fit(X, y)

prediccio_Tree = model_Decision_Tree.predict(X_2025)
print("Ventas reales:", y_2025.to_numpy()[:12])
print("Predicciones: ", prediccio_Tree[:12].round(2))

print("MAE:", mean_absolute_error(y_2025, prediccio_Tree))
print("RMSE:", np.sqrt(mean_squared_error(y_2025, prediccio_Tree)))
print("R²:", r2_score(y_2025, prediccio_Tree))

Ventas reales: [ 60  76  57 121  83 109 102 104  81  76  68  68]
Predicciones:  [ 47. 106.  54.  82.  82.  82.  93.  96. 106.  84.  58.  91.]
MAE: 21.13425925925926
RMSE: 27.64331343702809
R²: 0.6516646010853994


# 3. SVR

In [ ]:
from sklearn.svm import SVR

model_svr = SVR()
model_svr.fit(X, y)
prediccio_svr = model_svr.predict(X_2025)
print("Ventas reales:", y_2025.to_numpy()[:12])
print("Predicciones: ", prediccio_svr[:12].round(2))

print("MAE:", mean_absolute_error(y_2025, prediccio_svr))
print("RMSE:", np.sqrt(mean_squared_error(y_2025, prediccio_svr)))
print("R²:", r2_score(y_2025, prediccio_svr))